In [8]:
#### Analyze the temporal trend of VZ (mangrove) expansion ######
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import imageio
import imageio.v3 as iio  # For creating GIFs
import rasterio
from matplotlib import rcParams
import matplotlib.patches as patches
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from shapely.geometry import LineString
import geopandas as gpd
from scipy.interpolate import UnivariateSpline
import contextily as ctx
import plotly.express as px  # For alluvial (Sankey) plots
from shapely.geometry import Polygon, MultiPolygon
from rasterio.mask import mask
import pandas as pd
import geopandas as gpd
from shapely.geometry import mapping
from collections import Counter
from shapely.ops import split
from matplotlib.transforms import Affine2D
from matplotlib.patches import Rectangle
from rasterio.plot import show
import plotly.io as pio
import plotly.graph_objects as go
from rasterio.transform import xy
import glob
import re

In [9]:

# ----------------------------------------------------
# Folder
# ----------------------------------------------------
folder = "/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/Roy/Subregion_new/Region_0"

files = sorted(glob.glob(os.path.join(folder, "*.tif")))

label_map = {
    1: "VZ",
    2: "WZ",
    3: "TZ"
}

records = []

# ----------------------------------------------------
# Loop through every raster
# ----------------------------------------------------
for f in files:

    # extract year from filename
    year = int(re.search(r"\d{4}", os.path.basename(f)).group())

    with rasterio.open(f) as src:

        img = src.read(1)
        transform = src.transform

        for cls, name in label_map.items():

            rows, cols = np.where(img == cls)

            if len(rows) == 0:
                continue

            # centroid in pixel coordinates
            row_mean = rows.mean()
            col_mean = cols.mean()

            # convert to map coordinates
            x, y = xy(transform, row_mean, col_mean)

            records.append({
                "Year": year,
                "Class": name,
                "Centroid_X": x,
                "Centroid_Y": y,
                "Pixels": len(rows)
            })

centroids = pd.DataFrame(records)

print(centroids.head())

centroids.to_csv("class_centroids.csv", index=False)

   Year Class     Centroid_X    Centroid_Y  Pixels
0  1984    VZ  536480.141353  2.790019e+06   35369
1  1984    WZ  541630.863001  2.792095e+06    9379
2  1984    TZ  539951.939147  2.792559e+06   38450
3  1985    VZ  536327.333700  2.789714e+06   29316
4  1985    WZ  541795.582241  2.792227e+06   13729


In [10]:
import numpy as np

centroids = centroids.sort_values(["Class", "Year"])

centroids["dX"] = centroids.groupby("Class")["Centroid_X"].diff()
centroids["dY"] = centroids.groupby("Class")["Centroid_Y"].diff()

centroids["Distance"] = np.sqrt(
    centroids["dX"]**2 +
    centroids["dY"]**2
)

# direction (0° = east, 90° = north)
centroids["Direction_deg"] = np.degrees(
    np.arctan2(
        centroids["dY"],
        centroids["dX"]
    )
)

In [11]:
summary = []

for c in centroids["Class"].unique():

    df = centroids[centroids.Class == c]

    dx = df.iloc[-1]["Centroid_X"] - df.iloc[0]["Centroid_X"]
    dy = df.iloc[-1]["Centroid_Y"] - df.iloc[0]["Centroid_Y"]

    distance = np.sqrt(dx**2 + dy**2)

    angle = np.degrees(np.arctan2(dy, dx))

    summary.append({
        "Class": c,
        "Net_dx_m": dx,
        "Net_dy_m": dy,
        "Net_distance_m": distance,
        "Direction_deg": angle
    })

summary = pd.DataFrame(summary)

print(summary)

  Class    Net_dx_m    Net_dy_m  Net_distance_m  Direction_deg
0    TZ  732.002512  251.640321      774.048144      18.971420
1    VZ  282.220639  370.931222      466.088468      52.734541
2    WZ -908.107144   64.376491      910.386137     175.945037
